In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd

from mothernet.utils import load_ihdp_data, generate_data

In [ ]:

data, exclude_cols, y_col_name = load_ihdp_data(ihdp_path = Path("/Users/vzuev/Documents/git/git_other/CEVAE/datasets/IHDP"))
data

In [ ]:
data_x, data_y = data.drop(columns=[*exclude_cols, y_col_name]), data[y_col_name]
data_x  # categorical features are already encoded as ordinals

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

train_x, test_x, train_y, test_y = train_test_split(data_x, data_y, random_state=0)
train_y = pd.DataFrame(train_y)

In [ ]:
from mothernet.prediction.mothernet_additive import MotherNetAdditiveRegressor
from mothernet.utils import get_mn_model

model_path = get_mn_model("baam_Daverage_l1e-05_maxnumclasses0_nsamples500_numfeatures10_yencoderlinear_05_08_2024_03_04_01_epoch_40.cpkt")
reg = MotherNetAdditiveRegressor(device="cpu", path=model_path)

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder

prep = make_column_transformer((OrdinalEncoder(), train_x.dtypes == "category"), remainder='passthrough')
train_x_pre = prep.fit_transform(train_x)

In [ ]:
ss = StandardScaler().fit(train_y)
reg.fit(train_x_pre, ss.transform(train_y))  # ~14.2 sec on CPU for 747 data points

In [ ]:
import matplotlib.pyplot as plt

y_pred = reg.predict(prep.transform(test_x))
plt.plot(test_y, ss.inverse_transform(y_pred.reshape(-1, 1)), 'o')
plt.xlabel("test_y")
plt.ylabel("y_pred")